# 07 - Production Readiness and Monitoring

This notebook is your release gate for model deployment.


In [ ]:
# Cell 0: Setup
from __future__ import annotations

from datetime import UTC, datetime
import json
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent.parent

metrics_path = ROOT / 'ml' / 'models' / 'churn_reorder_metrics.json'
baseline_path = ROOT / 'ml' / 'models' / 'churn_drift_baseline.json'
scoring_path = ROOT / 'ml' / 'data' / 'churn_scoring_latest.csv'
dataset_path = ROOT / 'ml' / 'data' / 'churn_training_dataset.csv'

for p in [metrics_path, baseline_path, scoring_path, dataset_path]:
    print(p.name, 'exists=', p.exists())


In [ ]:
# Cell 1: Preflight checks
metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
drift = json.loads(baseline_path.read_text(encoding='utf-8'))
dataset = pd.read_csv(dataset_path)

checks = []
checks.append(('test_roc_auc', metrics['test_metrics']['roc_auc'] >= 0.60, metrics['test_metrics']['roc_auc']))
checks.append(('test_pr_auc', metrics['test_metrics']['pr_auc'] >= 0.40, metrics['test_metrics']['pr_auc']))

train_rate = metrics['class_balance']['train_positive_rate']
test_rate = metrics['class_balance']['test_positive_rate']
checks.append(('class_balance_gap', abs(train_rate - test_rate) <= 0.20, abs(train_rate - test_rate)))
checks.append(('dataset_rows', len(dataset) >= 200, len(dataset)))
checks.append(('selected_feature_count', len(metrics['selected_features']) >= 5, len(metrics['selected_features'])))

preflight = pd.DataFrame(checks, columns=['check', 'pass', 'value'])
display(preflight)


In [ ]:
# Cell 2: Production drift monitoring (PSI)
scored = pd.read_csv(scoring_path)
cur = pd.to_numeric(scored['reorder_probability_30d'], errors='coerce').dropna().to_numpy()

bin_edges = np.array(drift['bin_edges'])
expected = np.array(drift['test_distribution'])
counts, _ = np.histogram(cur, bins=bin_edges)
actual = counts / max(1, counts.sum())

def psi(exp, act):
    eps = 1e-9
    exp = np.clip(exp, eps, None)
    act = np.clip(act, eps, None)
    return float(np.sum((act - exp) * np.log(act / exp)))

psi_value = psi(expected, actual)
high_rate = float((scored['risk_bucket'] == 'high').mean())

monitor = {
    'generated_at_utc': datetime.now(UTC).isoformat(),
    'psi': psi_value,
    'high_risk_rate': high_rate,
    'psi_alert': psi_value >= 0.25,
    'high_risk_alert': high_rate >= 0.45,
}
monitor


In [ ]:
# Cell 3: Save release and monitoring reports
passed = bool(preflight['pass'].all())
release_report = {
    'generated_at_utc': datetime.now(UTC).isoformat(),
    'passed': passed,
    'release_blocked': not passed,
    'checks': preflight.to_dict(orient='records'),
}

release_json = ROOT / 'ml' / 'models' / 'release_preflight_report.json'
release_json.write_text(json.dumps(release_report, indent=2), encoding='utf-8')

monitor_json = ROOT / 'ml' / 'data' / 'reports' / 'churn' / 'production_monitoring.json'
monitor_json.parent.mkdir(parents=True, exist_ok=True)
monitor_json.write_text(json.dumps(monitor, indent=2), encoding='utf-8')

print('Saved:', release_json)
print('Saved:', monitor_json)
print('Release passed:', passed)


## Next Notebook

Proceed to `08_causal_production_decisioning.ipynb`.
